##### next_day()

- Returns the `next` given `weekday` after a date.
- Returns the `first date` which is `later than` the `value of the date column`.

**Important:**
- It `returns` the **next occurrence, not same day**.
- If `date` is `Monday` and you ask for `Monday` → it gives `next week's Monday`.

|    order_date    |   day_name     |   next_monday   |  day_name  |
|------------------|----------------|-----------------|------------|
|    2026-02-16	   |     Monday	    |   2026-02-23	  |   Monday   |

**Syntax**

     next_day(expr, dayOfWeek)

|     Argument      |                     Description                                        |
|-------------------|------------------------------------------------------------------------|
|  **expr**         | A **DATE expression**.                                                 |
|  **dayOfWeek**    | A **STRING** expression **identifying a day of the week**.             |
|  **Return**       | **DATE**.                                                              |

- The `next_day` function takes an **optional second argument**, which specifies the **day of the week** to return.  
- If **no day of the week** is specified, the function returns the **next day**.
- **dayOfWeek** must be one of the following **(case insensitive)**:
  - `'SU', 'SUN', 'SUNDAY'`
  - `'MO', 'MON', 'MONDAY'`
  - `'TU', 'TUE', 'TUESDAY'`
  - `'WE', 'WED', 'WEDNESDAY'`
  - `'TH', 'THU', 'THURSDAY'`
  - `'FR', 'FRI', 'FRIDAY'`
  - `'SA', 'SAT', 'SATURDAY'`

**Content:**
- `How to find Next Sunday?`
- `How to find Next Monday?`
- `How to find Next Friday Using with current_date()?`
- `How to find Next Day from a Date Column?`
  - `Next Day with a Specific Day of the Week`
  - `Multiple Weekdays`
- `How to Find Next Day from a Timestamp Column?`
- `How to Find Next Day Using in Spark SQL?`

In [0]:
from pyspark.sql.functions import col, to_date, next_day, date_format, current_date

##### 1) How to Find Next Sunday?

In [0]:
data = [("2026-03-10",), ("2026-03-11",), ("2026-03-12",), ("2026-03-13",), ("2026-03-14",)]
df1 = spark.createDataFrame(data, ["date"])

df1.select(
    col("date"),
    next_day(col("date"), "Sun").alias("next_sunday")
).display()

date,next_sunday
2026-03-10,2026-03-15
2026-03-11,2026-03-15
2026-03-12,2026-03-15
2026-03-13,2026-03-15
2026-03-14,2026-03-15


##### 2) How to Find Next Monday?

In [0]:
df1.select(
    col("date"),
    next_day("date", "Mon").alias("next_monday")
).display()

date,next_monday
2026-03-10,2026-03-16
2026-03-11,2026-03-16
2026-03-12,2026-03-16
2026-03-13,2026-03-16
2026-03-14,2026-03-16


##### 3) How to Find `Next Friday` Using with `current_date()`?

In [0]:
df2 = spark.range(1) \
    .select(current_date().alias("today"))

df2.select(
    "today",
    next_day("today", "Fri").alias("next_friday")
).display()

today,next_friday
2026-03-14,2026-03-20


##### 4) How to find `Next Day` from a `Date Column`?

In [0]:
# Create a sample DataFrame with a date column (ensure the column is a date type)
df_day = spark.createDataFrame([
    (1, "2026-02-16"),
    (2, "2025-12-31"),
    (3, "2025-01-02"),
    (4, "2025-01-10"),
    (5, "2025-01-28")
], ["id", "order_date"])

# Convert the date column to a DateType
df_day = df_day.withColumn('order_date', to_date(col('order_date')))
display(df_day)

id,order_date
1,2026-02-16
2,2025-12-31
3,2025-01-02
4,2025-01-10
5,2025-01-28


##### `a) Next Day with a Specific Day of the Week`

In [0]:
df_next = df_day \
    .withColumn('src_day_name', date_format('order_date', 'EEEE')) \
    .withColumn("next_monday", next_day(col("order_date"), "Monday")) \
    .withColumn('trg_day_name', date_format('next_monday', 'EEEE'))

display(df_next)
df_next.printSchema()

id,order_date,src_day_name,next_monday,trg_day_name
1,2026-02-16,Monday,2026-02-23,Monday
2,2025-12-31,Wednesday,2026-01-05,Monday
3,2025-01-02,Thursday,2025-01-06,Monday
4,2025-01-10,Friday,2025-01-13,Monday
5,2025-01-28,Tuesday,2025-02-03,Monday


root
 |-- id: long (nullable = true)
 |-- order_date: date (nullable = true)
 |-- src_day_name: string (nullable = true)
 |-- next_monday: date (nullable = true)
 |-- trg_day_name: string (nullable = true)



- `16th Feb 2026` is `Monday`, so `next Monday` is `23rd Feb 2026`.
- `31st Dec 2025` is `Wednesday`, so `next Monday` is `1st May 2025`.
- `2nd Jan 2025` is `Thursday`, so `next Monday` is `6th Jan 2025`.
- `10th Jan 2025` is `Friday`, so `next Monday` is `13th Jan 2025`.
- `28th Jan 2025` is `Tuesday`, so `next Monday` is `3rd Feb 2025`.

##### `b) Multiple Weekdays`

In [0]:
df_mlt = spark.createDataFrame([("2026-02-18",), ("2025-01-03",), ("2026-01-15",), ("2025-05-11",), ("2025-08-18",)], ["input_date"])
display(df_mlt)

input_date
2026-02-18
2025-01-03
2026-01-15
2025-05-11
2025-08-18


In [0]:
from pyspark.sql.functions import current_date

df_mlt_final = df_mlt.select(
    col("input_date"),
    date_format('input_date', 'EEEE').alias("1st_trg_day_name"),
    next_day(col("input_date"), "Mon").alias("next_monday"),
    date_format('next_monday', 'EEEE').alias("2nd_trg_day_name"),
    next_day(col("input_date"), "Fri").alias("next_friday"),
    date_format('next_friday', 'EEEE').alias("3rd_trg_day_name"),
    next_day(current_date(), "Sat").alias("next_saturday"),
    date_format('next_saturday', 'EEEE').alias("4th_trg_day_name"),
)

display(df_mlt_final)

input_date,1st_trg_day_name,next_monday,2nd_trg_day_name,next_friday,3rd_trg_day_name,next_saturday,4th_trg_day_name
2026-02-18,Wednesday,2026-02-23,Monday,2026-02-20,Friday,2026-03-21,Saturday
2025-01-03,Friday,2025-01-06,Monday,2025-01-10,Friday,2026-03-21,Saturday
2026-01-15,Thursday,2026-01-19,Monday,2026-01-16,Friday,2026-03-21,Saturday
2025-05-11,Sunday,2025-05-12,Monday,2025-05-16,Friday,2026-03-21,Saturday
2025-08-18,Monday,2025-08-25,Monday,2025-08-22,Friday,2026-03-21,Saturday


##### 5) How to Find `Next Day` from a `Timestamp Column`?

In [0]:
# Create a DataFrame with a timestamp column
data = [("2022-07-25 12:00:00",), ("2023-01-01 08:00:00",), ("2022-12-31 23:59:59",), ("2024-01-01 22:55:46",), ("2025-05-18 19:43:39",)]
df_ts = spark.createDataFrame(data, ["timestamp"])
display(df_ts)

timestamp
2022-07-25 12:00:00
2023-01-01 08:00:00
2022-12-31 23:59:59
2024-01-01 22:55:46
2025-05-18 19:43:39


In [0]:
from pyspark.sql.functions import to_timestamp

# Convert the timestamp column to a TimestampType
df_ts = df_ts.withColumn("timestamp", to_timestamp("timestamp"))

# Get the next day from the timestamp column
df_ts_next = df_ts \
  .withColumn('src_day_name', date_format('timestamp', 'EEEE')) \
  .withColumn("next_monday", next_day(col("timestamp"), "Monday")) \
  .withColumn('trg_day_name', date_format('next_monday', 'EEEE'))

# Display the result
display(df_ts_next)

timestamp,src_day_name,next_monday,trg_day_name
2022-07-25T12:00:00.000Z,Monday,2022-08-01,Monday
2023-01-01T08:00:00.000Z,Sunday,2023-01-02,Monday
2022-12-31T23:59:59.000Z,Saturday,2023-01-02,Monday
2024-01-01T22:55:46.000Z,Monday,2024-01-08,Monday
2025-05-18T19:43:39.000Z,Sunday,2025-05-19,Monday


##### 6) How to Find `Next Day` Using in `Spark SQL`?

In [0]:
%sql
SELECT next_day('2026-03-10','Sun') AS next_sunday;

next_sunday
2026-03-15
